# Pydantic Fundamentals

## Validate without pydantic

In [1]:
class Person:
    def __init__(self, name: str, gender: str, age: int) -> None:
        self.name = name
        self.gender = gender
        self.age = age
        
    def __repr__(self):
        return f"Person({self.name}, {self.gender}, {self.age})"
        
person1 = Person(name="Ludvig", gender="Male", age=35)
person1

Person(Ludvig, Male, 35)

In [2]:
# Issue: Even with type hints, there's nothing preventing this to work
person2 = Person(name=1138, gender=True, age="-13")  
person2

Person(1138, True, -13)

In [3]:
# Validate in constructor
class Person:
    def __init__(self, name: str, gender: str, age: int) -> None:
        if not isinstance(name, str):
            raise TypeError(f"Name must be of type str, not {type(name)}")
        self.name = name
        
        self.gender = gender
        
        if not isinstance(age, int):
            raise TypeError(f"Age must be of type int, not {type(age)}")
        if not 0 <= age < 125:
            raise ValueError(f"Age must be between 0 and 124, not {age}")
        self.age = age
        
    def __repr__(self):
        return f"Person({self.name}, {self.gender}, {self.age})"

try:        
    person3 = Person(name="Max", gender="Male", age=-35)
except ValueError as err:
    print(err)

Age must be between 0 and 124, not -35


In [4]:
# Issue: When a new value is assigned, validation in constructor doesn't run
person3 = Person(name="Max", gender="Male", age=35)
person3.age = -1
person3  

Person(Max, Male, -1)

In [5]:
# Validate using getter and setter
class Person:
    def __init__(self, name: str, gender: str, age: int) -> None:
        if not isinstance(name, str):
            raise TypeError(f"Name must be of type str, not {type(name)}")
        self.name = name
        self.gender = gender
        self.age = age
        
    @property
    def age(self):
        return self._age
    
    @age.setter
    def age(self, age):
        if not isinstance(age, int):
            raise TypeError(f"Age must be of type int, not {type(age)}")
        if not 0 <= age < 125:
            raise ValueError(f"Age must be between 0 and 124, not {age}")
        self._age = age
        
    def __repr__(self):
        return f"Person({self.name}, {self.gender}, {self.age})"

try:        
    person3 = Person(name="Max", gender="Male", age=35)
    person3.age = -1
except ValueError as err:
    print(err)

Age must be between 0 and 124, not -1


## Validate using pydantic

In [6]:
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    gender: str
    age: int
    
person4 = Person(name="Alex", gender="Male", age=58)
person4

Person(name='Alex', gender='Male', age=58)

In [7]:
from pydantic import ValidationError

try:
    person5 = Person(name=1509, gender="Male", age=27)
except ValidationError as err:
    print(err)

1 validation error for Person
name
  Input should be a valid string [type=string_type, input_value=1509, input_type=int]
    For further information visit https://errors.pydantic.dev/2.5/v/string_type


In [8]:
# str "26" is coerced into 26
person6 = Person(name="Emanuel", gender="Male", age="26")
person6

Person(name='Emanuel', gender='Male', age=26)

In [9]:
# Issue: Values can be changed afterwards
person6.age = "Eighteen"
person6

Person(name='Emanuel', gender='Male', age='Eighteen')

In [ ]:
# Validate using ConfigDict
from pydantic import ConfigDict

class Person(BaseModel):
    name: str
    gender: str
    age: int
    
    model_config = ConfigDict(validate_assignment=True)

person7 = Person(name="Jonas", gender="Male", age=15)
    
try:
    person7.age = "Three"
except ValidationError as err:
    print(err)

1 validation error for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='Three', input_type=str]
    For further information visit https://errors.pydantic.dev/2.5/v/int_parsing


In [12]:
try:
    Person(name=51, gender=15, age=3)
except ValidationError as err:
    print(err)

2 validation errors for Person
name
  Input should be a valid string [type=string_type, input_value=51, input_type=int]
    For further information visit https://errors.pydantic.dev/2.5/v/string_type
gender
  Input should be a valid string [type=string_type, input_value=15, input_type=int]
    For further information visit https://errors.pydantic.dev/2.5/v/string_type


In [14]:
# Add validation conditions
from pydantic import Field
from typing import Literal

class Person(BaseModel):
    name: str
    gender: Literal["F", "M"]
    age: int = Field(gt=-1, lt=125)
    
    model_config = ConfigDict(validate_assignment=True)
    
try:
    Person(name="William", gender="Male", age=-3)
except ValidationError as err:
    print(err)

2 validation errors for Person
gender
  Input should be 'F' or 'M' [type=literal_error, input_value='Male', input_type=str]
    For further information visit https://errors.pydantic.dev/2.5/v/literal_error
age
  Input should be greater than -1 [type=greater_than, input_value=-3, input_type=int]
    For further information visit https://errors.pydantic.dev/2.5/v/greater_than


## Serialization and deserialization

In [ ]:
# Serialization
person7.model_dump()

{'name': 'Jonas', 'gender': 'Male', 'age': 15}